# Creating agentic workflows in LlamaIndex

A workflow in LlamaIndex provides a structured way to organize your code into sequential and manageable steps. A workflow is created by defining `steps` which are triggered by `events` and themselves emit `events` to trigger further steps.

In [48]:
# !pip install llama-index-utils-workflow

### Basic workflow

In [49]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step

class MyWorkflow(Workflow):
    @step # decorator
    async def my_step(self, ev:StartEvent) -> StopEvent: # async function, (self, ev: StartEvent) -> StopEvent: take a start event, return a stop event
        return StopEvent(result = "Hello, world!")
    

w = MyWorkflow(timeout = 10, verbose = True) # verbose = False, no need to print the log
result = await w.run()
result

[tick] add: StartEvent()
[my_step:0] started from StartEvent
[result] StopEvent(result='Hello, world!')
[my_step:0] complete with StopEvent


'Hello, world!'

### Multiple steps

Type hinting is important

In [50]:
from llama_index.core.workflow import Event

class ProcessingEvent(Event):
    intermediate_result: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        return ProcessingEvent(intermediate_result = "Step 1 complete")
    
    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result = final_result)
    
w = MultiStepWorkflow(timeout = 10, verbose = True)
result = await w.run()
retult

[tick] add: StartEvent()
[step_one:0] started from StartEvent
[step_one:0] complete with ProcessingEvent
[tick] add: ProcessingEvent(intermediate_result='Step 1 complete')
[step_two:0] started from ProcessingEvent
[result] StopEvent(result='Finished processing: Step 1 complete')
[step_two:0] complete with StopEvent


'Hello, world!'

### Loops and Branches

In [51]:
from llama_index.core.workflow import Event
import random

class ProcessingEvent(Event):
    intermediate_result: str

class LoopEvent(Event):
    loop_output: str

class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0,1) ==0:
            print("Bad thing happened")
            return ProcessingEvent(intermediate_result = "First step complete.")
        else:
            print("Good thing happened")
            return LoopEvent(loop_output = "Loop")
    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result = final_result)
    
w = MultiStepWorkflow(verbose = True)
result = await w.run()
result


[tick] add: StartEvent()
[step_one:0] started from StartEvent
Bad thing happened
[step_one:0] complete with ProcessingEvent
[tick] add: ProcessingEvent(intermediate_result='First step complete.')
[step_two:0] started from ProcessingEvent
[result] StopEvent(result='Finished processing: First step complete.')
[step_two:0] complete with StopEvent


'Finished processing: First step complete.'

In [52]:
# visualize the workflow
from llama_index.utils.workflow import draw_all_possible_flows

w = MultiStepWorkflow(verbose = True)
draw_all_possible_flows(w)

workflow_all_flows.html


### Statemanagement

We use `Context` to manage the state.

```pasudocode
from llama_index.core.workflow import Context, StartEvent, StopEvent

@step
async def query(self, ctx: Context, ev: StartEvent) -> StopEvent:
    await ctx.store.set("query", "What is the capital of France?")

    #do something with context and event
    val = ...

    # retrieve query from the context
    query = await ctx.store.get("query")

    return StopEvent(result = val)


```

### Automating workflows with Multi-Agent Workflows

Each agent can then:
- Handle the request directly using the tools
- Handoff to another agent better suited for the task
- Return a response to the user

One agent must be designed as the root agent in the Agent Workflow constructor. When the message comes in, it is first routed to the root agent.

In [53]:
from llama_index.core.agent.workflow import AgentWorkflow, ReActAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(
    api_base = "http://localhost:1234/v1",
    api_key="lm-studio",
    model="o1",
)

# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

# we can pass functions directly without FunctionTool -- the fn/docstring are parsed for the name/description
multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

# Create the workflow
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
)

# Run the system
response = await workflow.run(user_msg="Can you add 5 and 3?")

In [54]:
print(str(response))

5 plus 3 is 8.


In [55]:
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='5 plus 3 is 8.')]), structured_response=None, current_agent_name='add_agent', raw={'id': 'chatcmpl-1d5t3ntkc71nidw7buwh2', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'created': 1783328335, 'model': 'gemma-4-26b-a4b-it-qat', 'object': 'chat.completion.chunk', 'service_tier': None, 'system_fingerprint': 'gemma-4-26b-a4b-it-qat', 'usage': None}, tool_calls=[ToolCallResult(tool_name='handoff', tool_kwargs={'to_agent': 'add_agent', 'reason': 'The user wants to add two numbers (5 and 3), which requires an addition agent.'}, tool_id='43741670-0954-4c8b-948d-8c8c8d2613f2', tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Agent add_agent is now handling the request due to the following reason: The user wants to a

In [56]:
from llama_index.core.workflow import Context

# Define some tools
async def add(ctx: Context, a: int, b: int) -> int:
    """Add two numbers."""
    # update our count
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a + b

async def multiply(ctx: Context, a: int, b: int) -> int:
    """Multiply two numbers."""
    # update our count
    cur_state = await ctx.store.get("state")
    cur_state["num_fn_calls"] += 1
    await ctx.store.set("state", cur_state)

    return a * b

multiply_agent = ReActAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="A helpful assistant that can use a tool to multiply numbers.",
    tools=[multiply],
    llm=llm,
)

addition_agent = ReActAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="A helpful assistant that can use a tool to add numbers.",
    tools=[add],
    llm=llm,
)

workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent",
    initial_state={"num_fn_calls": 0},
    state_prompt="Current state: {state}. User message: {msg}",
)

# run the workflow with context
ctx = Context(workflow)
response = await workflow.run(user_msg="Can you add 5 and 3?", ctx=ctx)

# pull out and inspect the state
state = await ctx.store.get("state")
print(state["num_fn_calls"])

1


In [57]:
str(response)

'5 + 3 is 8.'

In [47]:
ctx.to_dict()

{'version': 2,
 'state': {'store_type': 'in_memory',
  'state_type': 'DictState',
  'state_module': 'workflows.context.state_store',
  'state_data': {'_data': {'memory': '{"__is_component": true, "value": {"chat_store": {"store": {"chat_history": [{"role": "user", "additional_kwargs": {}, "blocks": [{"block_type": "text", "text": "Can you add 5 and 3?"}]}, {"role": "assistant", "additional_kwargs": {}, "blocks": [{"block_type": "text", "text": "Thought: The current language of the user is: English. I need to use a tool to help me answer the question. Since the user wants to add numbers and I only have a multiply tool, I should hand off to the add_agent.\\nAction: handoff\\nAction Input: {\'to_agent\': \'add_agent\', \'reason\': \'The user wants to perform an addition operation, which is handled by the add_agent.\'}\\nObservation: Agent add_agent is now handling the request due to the following reason: The user wants to perform an addition operation, which is handled by the add_agent..\